# Production Adaptive Synchronization Model

This Jupyter Notebook upgrades the single Random Forest classifier into a production-grade ML pipeline for adaptive data synchronization.

### Key Enhancements & Architecture:
1. **`HistGradientBoostingClassifier`**: Replaces single Random Forest with an ensemble model (similar to XGBoost/LightGBM) optimized for fast inference.
2. **Feature Engineering & Interaction Terms**: Computes dynamic interactions like `bandwidth_per_latency` and `signal_battery_product`.
3. **Telemetry Normalization**: Handles client key variations (`SignalStrengthPercent`, `LatencyMs`, `NetworkT`, snake_case, etc.).
4. **Threshold Optimization & Validation**: Dynamically optimizes probability thresholds via precision-recall curves and provides 5-Fold Stratified Cross-Validation metrics.
5. **Real-time Safety Policy Rules**: Integrates ML decision probabilities with hard battery and budget safety guardrails.

In [78]:
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import precision_recall_curve

MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / "adaptive_sync_hgbc.joblib"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "signal_strength", 
    "bandwidth_kbps", 
    "latency_ms", 
    "battery_percent", 
    "queue_size", 
    "is_wifi",
    "bandwidth_per_latency",
    "signal_battery_prod"
]

## 1. Feature Engineering & Key Normalization

In [79]:
def normalize_telemetry(context: Dict[str, Any]) -> Dict[str, Any]:
    """Standardizes client payload dictionary keys across SDK variants."""
    key_map = {
        "SignalStrengthPercent": "signal_strength",
        "BandwidthKbps": "bandwidth_kbps",
        "LatencyMs": "latency_ms",
        "LatencyM": "latency_ms",
        "BatteryPercent": "battery_percent",
        "NetworkType": "network_type",
        "NetworkT": "network_type",
        "QueueSize": "queue_size",
        "DataBudgetKb": "data_budget_kb",
    }
    normalized = {}
    for k, v in context.items():
        standard_key = key_map.get(k, k.lower())
        normalized[standard_key] = v
    return normalized

def extract_features(context: Dict[str, Any]) -> pd.DataFrame:
    """Extracts base signals and calculates interaction feature columns."""
    ctx = normalize_telemetry(context)
    
    signal = float(ctx.get("signal_strength") if ctx.get("signal_strength") is not None else 50.0)
    bandwidth = float(ctx.get("bandwidth_kbps") if ctx.get("bandwidth_kbps") is not None else 500.0)
    latency = float(ctx.get("latency_ms") if ctx.get("latency_ms") is not None else 250.0)
    battery = float(ctx.get("battery_percent") if ctx.get("battery_percent") is not None else 50.0)
    queue_size = int(ctx.get("queue_size", 0))
    
    net_type = str(ctx.get("network_type", "")).lower().strip().replace("-", "")
    is_wifi = 1 if net_type in ("wifi", "wi-fi") else 0
    
    # Engineered Interaction Features
    bandwidth_per_latency = bandwidth / (latency + 1.0)
    signal_battery_prod = (signal / 100.0) * (battery / 100.0)
    
    row = {
        "signal_strength": signal,
        "bandwidth_kbps": bandwidth,
        "latency_ms": latency,
        "battery_percent": battery,
        "queue_size": queue_size,
        "is_wifi": is_wifi,
        "bandwidth_per_latency": bandwidth_per_latency,
        "signal_battery_prod": signal_battery_prod,
    }
    return pd.DataFrame([row], columns=FEATURES)

## 2. Gradient Boosting ML Model Pipeline Class

In [80]:
class EnterpriseSyncModel:
    """Production Gradient Boosting Model Wrapper with Cross-Validation and Model Persistence."""
    
    def __init__(self, model_path: Path = MODEL_PATH):
        self.model_path = model_path
        self.model: Optional[HistGradientBoostingClassifier] = None
        self.best_threshold: float = 0.50
        self._load_model_if_exists()
        
    def _load_model_if_exists(self):
        if self.model_path.exists():
            try:
                payload = joblib.load(self.model_path)
                if isinstance(payload, dict) and "model" in payload:
                    self.model = payload["model"]
                    self.best_threshold = payload.get("threshold", 0.50)
                else:
                    self.model = payload
            except Exception as e:
                print(f"Warning: Could not load cached model: {e}")

    def train(self, observations: List[Dict[str, Any]]) -> Dict[str, Any]:
        rows = []
        for item in observations:
            if item.get("sync_success") is None:
                continue
            feat_df = extract_features(item)
            row = feat_df.iloc[0].to_dict()
            row["target"] = 1 if item["sync_success"] else 0
            rows.append(row)

        if len(rows) < 20 or len({r["target"] for r in rows}) < 2:
            return {
                "trained": False,
                "samples": len(rows),
                "message": "Insufficient data: Require at least 20 labelled records containing both successful and failed syncs."
            }

        df = pd.DataFrame(rows)
        X = df[FEATURES]
        y = df["target"]

        self.model = HistGradientBoostingClassifier(
            max_iter=150,
            learning_rate=0.05,
            max_leaf_nodes=31,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42
        )

        # Stratified K-Fold Cross-Validation
        cv = StratifiedKFold(n_splits=min(5, max(2, len(df) // 5)), shuffle=True, random_state=42)
        cv_results = cross_validate(self.model, X, y, cv=cv, scoring=["roc_auc", "f1", "precision", "recall"])
        
        self.model.fit(X, y)
        
        # Optimize Threshold for F1 Score
        probs = self.model.predict_proba(X)[:, 1]
        precisions, recalls, thresholds = precision_recall_curve(y, probs)
        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
        best_idx = np.argmax(f1_scores)
        self.best_threshold = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.50

        payload = {
            "model": self.model,
            "threshold": self.best_threshold,
            "features": FEATURES
        }
        joblib.dump(payload, self.model_path)

        return {
            "trained": True,
            "samples": len(rows),
            "cv_auc_mean": round(float(np.mean(cv_results["test_roc_auc"])), 4),
            "cv_f1_mean": round(float(np.mean(cv_results["test_f1"])), 4),
            "optimal_threshold": round(self.best_threshold, 4),
            "message": "HistGradientBoostingClassifier model trained and saved."
        }

    def predict_suitability(self, features_df: pd.DataFrame) -> Tuple[bool, float]:
        if self.model is None:
            return False, 0.0
        
        probs = self.model.predict_proba(features_df)[0]
        pos_idx = list(self.model.classes_).index(1) if 1 in self.model.classes_ else 1
        confidence = float(probs[pos_idx])
        is_suitable = confidence >= self.best_threshold
        return is_suitable, confidence

## 3. Policy Recommendation Logic

In [81]:
sync_engine = EnterpriseSyncModel()

def recommend_sync(context: Dict[str, Any]) -> Dict[str, Any]:
    """Production Decision Policy combining ML predictions with hard safety limits."""
    ctx = normalize_telemetry(context)
    queue_size = int(ctx.get("queue_size", 0))
    
    if queue_size <= 0:
        return {
            "strategy": "immediate",
            "should_sync": False,
            "reason": "No pending records.",
            "confidence": None,
        }

    features_df = extract_features(ctx)
    confidence = None
    suitable = None

    if sync_engine.model is not None:
        suitable, confidence = sync_engine.predict_suitability(features_df)

    # Transparent Rule Fallback
    signal = ctx.get("signal_strength")
    bandwidth = ctx.get("bandwidth_kbps")
    latency = ctx.get("latency_ms")
    battery = ctx.get("battery_percent")
    budget = ctx.get("data_budget_kb")
    net_type = str(ctx.get("network_type", "")).lower().strip().replace("-", "")
    wifi = net_type in ("wifi", "wi-fi")

    if suitable is None:
        suitable = (
            (signal is None or float(signal) >= 35.0)
            and (bandwidth is None or float(bandwidth) >= 128.0)
            and (latency is None or float(latency) <= 1200.0)
            and (battery is None or float(battery) >= 15.0)
        )

    # Safety Guardrails
    if battery is not None and float(battery) < 10.0:
        return {
            "strategy": "delayed",
            "should_sync": False,
            "reason": "Battery is critically low; wait for charging or a better battery level.",
            "confidence": confidence,
        }
    if not suitable:
        return {
            "strategy": "delayed",
            "should_sync": False,
            "reason": "Network conditions are not currently suitable; retry later.",
            "confidence": confidence,
        }

    if budget is not None and float(budget) < 64.0 and not wifi:
        return {
            "strategy": "partial",
            "should_sync": True,
            "max_records": min(queue_size, 5),
            "reason": "Limited data budget; synchronize a small batch.",
            "confidence": confidence,
        }
    if not wifi and queue_size > 50:
        return {
            "strategy": "partial",
            "should_sync": True,
            "max_records": 25,
            "reason": "Large queue on cellular data; send a bounded batch.",
            "confidence": confidence,
        }
    return {
        "strategy": "immediate",
        "should_sync": True,
        "max_records": queue_size,
        "reason": "Current conditions are suitable for synchronization.",
        "confidence": confidence,
    }

## 4. Run Training & Test Prediction

In [82]:
# Generate 150 synthetic observations for training
np.random.seed(42)
observations = []
for _ in range(15000):
    sig = np.random.uniform(10, 100)
    bw = np.random.uniform(50, 50000)
    lat = np.random.uniform(20, 1500)
    bat = np.random.uniform(5, 100)
    net = "Wi-Fi" if np.random.rand() > 0.4 else "Cellular"
    q_size = np.random.randint(1, 100)
    
    success = (sig > 30) and (bw > 200) and (lat < 1000) and (bat > 15)
    observations.append({
        "SignalStrengthPercent": sig,
        "BandwidthKbps": bw,
        "LatencyMs": lat,
        "BatteryPercent": bat,
        "NetworkT": net,
        "QueueSize": q_size,
        "sync_success": success
    })

# Execute Training
train_status = sync_engine.train(observations)
print("Training Status:", json.dumps(train_status, indent=2))

# Test Recommendation with sample row
sample_context = {
    "NetworkT": "Wi-Fi",
    "LatencyM": 186,
    "SignalStrengthPercent": 74,
    "BatteryPercent": 37,
    "BandwidthKbps": 22631,
    "QueueSize": 15
}


# Read your CSV dataset
dataset = pd.read_csv(r"D:\Downloads\Metrics\Metrics_train.csv")

# Generate recommendations for every row in the dataset
predictions = []
for row in dataset.to_dict(orient="records"):
    rec = recommend_sync(row)
    predictions.append({
        "should_sync": rec.get("should_sync"),
        "strategy": rec.get("strategy"),
        "max_records": rec.get("max_records"),
        "confidence": rec.get("confidence"),
        "reason": rec.get("reason"),
    })

# Convert predictions to DataFrame and concatenate with original dataset
results_df = pd.concat([dataset, pd.DataFrame(predictions)], axis=1)

# # Display the first 10 rows of results
# print(results_df.head(10))




Training Status: {
  "trained": true,
  "samples": 15000,
  "cv_auc_mean": 0.9996,
  "cv_f1_mean": 0.9964,
  "optimal_threshold": 0.5732,
  "message": "HistGradientBoostingClassifier model trained and saved."
}
